In [ ]:
import glob
import argparse
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import oggm
from oggm import utils

%matplotlib qt 

In [ ]:
pip install oggm

In [ ]:
# Setup oggm: important we want to use oggm's version 62 of all glacier geometries. This command will download them.
utils.get_rgi_dir(version='62')

In [ ]:
path_O1_shp = '//Users/emma/OGGM/rgi/RGIV62/00_rgi62_regions/00_rgi62_O1Regions.shp' # shp file of regional boundaries


# Import 20-bin gridded training dataset
metadata_file = "/media/maffe/nvme/glathida/glathida-3.1.0/glathida-3.1.0/data/metadata18_hmineq0.0_tmin20050000_mean_grid_20.csv"
glathida_rgis = pd.read_csv(metadata_file, low_memory=False)

In [ ]:
# See all RGI's regional boundaries
# See pag 8 at https://nsidc.org/sites/nsidc.org/files/technical-references/RGI_Tech_Report_V6.0.pdf
world = gpd.read_file(path_O1_shp)      # import as geopandas dataframe
print(world)
fig, ax = plt.subplots()
for n, rgi_series in enumerate(world['geometry']):
    x, y = rgi_series.exterior.xy
    min_x, max_x = min(x), max(x)
    min_y, max_y = min(y), max(y)
    ax.plot(x, y, c='g')
    ax.text(np.mean([min_x, max_x]), np.mean([min_y, max_y]), f"{world['RGI_CODE'].iloc[n]}", color='green', fontsize=9)
plt.show()

In [ ]:
# Let's select one region of interest, e.g. rgi=11 (Central Europe)

# Dataframe of glaciers for rgi=11. This dataset contains a lot of info, some of them are also used to create the
# training dataset. RGIId is the official name of glaciers according to the Randolph Glacier Inventory. 
# In the column geometry you get the geometries. 

rgi = 11
oggm_rgi_shp = glob.glob(f"/home/maffe/OGGM/rgi/RGIV62/{rgi}*/{rgi}*.shp")[0] # .shp file for rgi 11
oggm_rgi_glaciers = gpd.read_file(oggm_rgi_shp)                         # dataframe for rgi 11

In [ ]:
# Plot one specific glacier by passing the RGIId code.
# Example: RGI60-11.01450 is the Aletsch Glacier, the biggest in Central Europe (rgi=11).
# You under the geometry column you find the glacier external boundary (.external)
# and the glacier nunataks, if any (.interiors). Nunataks are the inner glacier portions that are deglaciated (=rock).
# We can also find the points in the training dataset for this glacier (not all glaciers contain measurements)

glacier_geometry = oggm_rgi_glaciers.loc[oggm_rgi_glaciers['RGIId']=='RGI60-11.01450']['geometry'].item()

# Get the measurements in the training dataset by passing the glacier id
glathida_rgis_aletsch = glathida_rgis.loc[glathida_rgis['RGIId']=='RGI60-11.01450']
print(f"We have {len(glathida_rgis_aletsch)} points in the training dataset")

fig, ax = plt.subplots()
exterior_ring = glacier_geometry.exterior # External geometry
ax.plot(*exterior_ring.xy, c='b')
glacier_nunataks_list = [nunatak for nunatak in glacier_geometry.interiors] # Nunataks (list may be empty if no nunataks)
for nunatak in glacier_nunataks_list:
    ax.plot(*nunatak.xy, c='k', lw=0.8)
ax.scatter(x=glathida_rgis_aletsch['POINT_LON'], y=glathida_rgis_aletsch['POINT_LAT'], s=20, c='orange', label='Training points')
ax.legend()
plt.show()

In [ ]:
# Plot the whole regional set of glaciers and all nunataks
# zoom in with matplotlib in interactive mode (%matplotlib qt ) to have a sense of all glacier and their nunataks

region11 = world.loc[12] # Regional boundary
fig, ax = plt.subplots()
for glacier_rgiid, glacier_geometry in zip(oggm_rgi_glaciers['RGIId'], oggm_rgi_glaciers['geometry']):

    exterior_ring = glacier_geometry.exterior
    glacier_nunataks_list = [nunatak for nunatak in glacier_geometry.interiors]

    ax.plot(*exterior_ring.xy, c='b')
    for nunatak in glacier_nunataks_list:
        ax.plot(*nunatak.xy, c='k', lw=0.8) # Plot glacier nunataks (if any)
ax.plot(*region11['geometry'].exterior.xy, c='g') # Regional boundary
ax.text(min(*region11['geometry'].exterior.xy[0])+1,
        max(*region11['geometry'].exterior.xy[1])-1, f'rgi {rgi}', c='g', fontsize=12)
plt.show()